In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import text, create_engine


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, array, array_remove, lit, array_contains, concat_ws, size, array_compact
from pyspark.ml.fpm import FPGrowth


In [3]:
# Khởi động spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FP-Growth Python")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.extraJavaOptions", "-Duser.timezone=Asia/Ho_Chi_Minh")
    .config("spark.executor.extraJavaOptions", "-Duser.timezone=Asia/Ho_Chi_Minh")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.4")
    .config("spark.sql.session.timeZone", "Asia/Ho_Chi_Minh")
    .getOrCreate()
)

spark._jvm.java.util.TimeZone.setDefault(
    spark._jvm.java.util.TimeZone.getTimeZone("Asia/Ho_Chi_Minh")
)


In [4]:
# DB config

load_dotenv("../../.env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

DWH = os.getenv("DB_SCHEMA_DWH")

jdbc_url = (
    f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
    "?options=-c%20TimeZone=Asia/Ho_Chi_Minh"
)

connection_properties = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}

# engine sqlalchemy
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

with engine.begin() as conn:
    conn.execute(
        text(f"truncate table {DWH}.fact_cal_rules_fp_growth_sell")
    )

df = spark.read.jdbc(
    url=jdbc_url,
    table=f"{DWH}.fact_txn_fp_growth_metrics",
    properties=connection_properties
)

df.show()


+-----------+----------+--------------------+--------------------+----------+-----------+------+-------------+--------------------+--------------+--------------------+--------------------+----------------+------------------------+----------------------------------------------+---------------------------------+-----------------------------------------------+-----------------------+-------------------+-------------------+------------------------------+--------------+-------------------+-------------------------------------+
|period_date|symbol_key|         close_price|   rule_weight_total|prediction|tomorrow_up|actual|actual_signal|close_price_gt_ma_50|ma_20_gt_ma_50|close_price_gt_ma_20|close_price_gt_ma_15|ema_12_gt_ema_26|macd_line_gt_signal_line|volume_gt_vol_ma_20_mul_1_5_and_return_1d_gt_0|return_1d_gt_0_and_return_3d_gt_0|close_price_gt_bb_upper_20_and_bb_width_20_tang|close_price_gt_high_10d|atr_14_gt_atr_ma_14|vol_ratio_20_gt_2_0|rsi_14_gte_30_and_rsi_14_lt_70|return_1d_gt_0|vol

In [5]:
#Rename feature columns

cols = df.columns

new_cols = (
    cols[:8] +
    [f"X{i}" for i in range(1, len(cols[8:]) + 1)]
)

df = df.toDF(*new_cols)
df.columns


['period_date',
 'symbol_key',
 'close_price',
 'rule_weight_total',
 'prediction',
 'tomorrow_up',
 'actual',
 'actual_signal',
 'X1',
 'X2',
 'X3',
 'X4',
 'X5',
 'X6',
 'X7',
 'X8',
 'X9',
 'X10',
 'X11',
 'X12',
 'X13',
 'X14',
 'X15',
 'X16']

In [6]:
# Drop unnecessary columns
from pyspark.sql.functions import when, col

df = df.drop(
        "close_price", 
        "rule_weight_total", 
        "actual_signal", 
        "tomorrow_up")

df = df.withColumn(
    "prediction",
    when(col("prediction") == "buy", 1)
    .otherwise(0)
)

df.columns


['period_date',
 'symbol_key',
 'prediction',
 'actual',
 'X1',
 'X2',
 'X3',
 'X4',
 'X5',
 'X6',
 'X7',
 'X8',
 'X9',
 'X10',
 'X11',
 'X12',
 'X13',
 'X14',
 'X15',
 'X16']

FP-Growth


In [7]:
from pyspark.ml.fpm import FPGrowth

#1. Filter sell
df_sell = df.filter(col("prediction") == 0)

#2. Create sell target
df_sell = df_sell.withColumn(
    "Sell",
    when(col("actual") == 1, 1).otherwise(0)
)

#3. Get rule cols
rule_cols = [c for c in df_sell.columns if c.startswith("X")]

#4. Create item column
item_exprs = [
    when(
        col(c) == 1, lit(c)
    ).otherwise(None)
    for c in rule_cols
]

target_expr = when(col("Sell") == 1, lit("Sell")).otherwise(None)

df_items = df_sell.withColumn(
    "items",
    array_compact(array(*item_exprs, target_expr))
).select("items")

#5. Run fp-growth
count = df_items.count()

fp = FPGrowth(
    itemsCol="items",
    minSupport=10 / count,
    minConfidence=0.6
)

model = fp.fit(df_items)

rules = model.associationRules


In [8]:
# Write DB
# Filter rules
rule_sell = (
    rules
    .filter(size(col("antecedent")) >= 3)
    .filter(array_contains(col("consequent"), "Sell"))
    .filter(~array_contains(col("antecedent"), "Sell"))
    .filter(col("confidence") >= 0.6)
)

rules_out = rule_sell.select(
    concat_ws(",", col("antecedent")).alias("antecedents"),
    concat_ws(",", col("consequent")).alias("consequents"),
    col("confidence"),
    col("lift")
)

rules_out.write.jdbc(
    url=jdbc_url,
    table=f"{DWH}.fact_cal_rules_fp_growth_sell",
    mode="append",
    properties=connection_properties
)

with engine.begin() as conn:
    conn.execute(text(f"call {DWH}.update_fact_cal_rules_fp_growth_prc();"))


In [9]:
print("count:", count)
print("rules:", rules.count())
print("rules_filtered:", rule_sell.count())
rules_out.show(20, False)


count: 43490
rules: 91516
rules_filtered: 352
+------------------------------+-----------+------------------+------------------+
|antecedents                   |consequents|confidence        |lift              |
+------------------------------+-----------+------------------+------------------+
|X9,X10,X7,X14,X3,X1,X13       |Sell       |0.7142857142857143|1.25122993975453  |
|X9,X10,X7,X8,X6,X3,X1,X13     |Sell       |0.7142857142857143|1.25122993975453  |
|X9,X10,X7,X6,X3,X1,X13        |Sell       |0.7142857142857143|1.25122993975453  |
|X9,X10,X7,X6,X4,X3,X1,X13     |Sell       |0.7142857142857143|1.25122993975453  |
|X9,X7,X15,X11,X6,X4,X3,X1     |Sell       |0.7142857142857143|1.25122993975453  |
|X9,X10,X15,X4,X1,X13          |Sell       |0.7142857142857143|1.25122993975453  |
|X9,X16,X11,X6,X4,X1,X13       |Sell       |0.8095238095238095|1.4180605983884673|
|X9,X7,X8,X14,X11,X6,X1        |Sell       |0.7142857142857143|1.25122993975453  |
|X9,X10,X7,X15,X8,X14,X3,X1,X13|Sell     